In [0]:
%pip install cryptography requests
dbutils.library.restartPython()

In [0]:
kalshi_api_key_id = dbutils.secrets.get("kalshi", "KALSHI_API_KEY_ID")
kalshi_private_key = dbutils.secrets.get("kalshi", "KALSHI_PRIVATE_KEY")

In [0]:
import time
import base64
import requests

from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding


# ============================================================
# Kalshi API config
# ============================================================

REST_HOST = "https://external-api.kalshi.com"
API_PREFIX = "/trade-api/v2"

SECRET_SCOPE = "kalshi"
API_KEY_ID_SECRET = "KALSHI_API_KEY_ID"

# Preferred if you store the private key as base64.
# Fallback supports raw PEM.
PRIVATE_KEY_B64_SECRET = "KALSHI_PRIVATE_KEY_B64"
PRIVATE_KEY_PEM_SECRET = "KALSHI_PRIVATE_KEY"


# ============================================================
# Simple secret helpers
# ============================================================

def get_secret(scope: str, key: str, required: bool = True) -> str | None:
    """
    Safely retrieve a Databricks secret.
    Returns None if required=False and the secret does not exist.
    """
    try:
        value = dbutils.secrets.get(scope=scope, key=key)
        if required and not value:
            raise ValueError(f"Secret {scope}/{key} is empty.")
        return value
    except Exception as e:
        if required:
            raise RuntimeError(f"Could not retrieve required secret: {scope}/{key}") from e
        return None


def normalize_pem(value: str) -> str:
    """
    Normalize a PEM key stored in Databricks secrets.
    Handles:
      - accidental wrapping quotes
      - escaped newlines: \\n
      - Windows newlines
      - extra blank lines / whitespace
    """
    value = value.strip()

    if (value.startswith('"') and value.endswith('"')) or (
        value.startswith("'") and value.endswith("'")
    ):
        value = value[1:-1].strip()

    value = (
        value.replace("\\r\\n", "\n")
             .replace("\\n", "\n")
             .replace("\r\n", "\n")
             .replace("\r", "\n")
    )

    lines = [line.strip() for line in value.split("\n") if line.strip()]
    return "\n".join(lines) + "\n"


def load_kalshi_private_key():
    """
    Loads Kalshi private key from Databricks secrets.

    Preferred:
      KALSHI_PRIVATE_KEY_B64 = base64-encoded PEM text

    Fallback:
      KALSHI_PRIVATE_KEY = raw PEM text
    """
    private_key_b64 = get_secret(
        SECRET_SCOPE,
        PRIVATE_KEY_B64_SECRET,
        required=False
    )

    if private_key_b64:
        print("Using base64 private key secret.")
        pem_text = base64.b64decode(private_key_b64).decode("utf-8")
    else:
        print("Using raw PEM private key secret.")
        pem_text = get_secret(
            SECRET_SCOPE,
            PRIVATE_KEY_PEM_SECRET,
            required=True
        )

    pem_text = normalize_pem(pem_text)

    lines = pem_text.splitlines()
    print("Private key first line:", repr(lines[0]) if lines else None)
    print("Private key last line:", repr(lines[-1]) if lines else None)
    print("Private key line count:", len(lines))

    if "BEGIN PUBLIC KEY" in lines[0]:
        raise ValueError(
            "You stored a public key, not a private key. "
            "Kalshi authentication requires the private key."
        )

    if "BEGIN ENCRYPTED PRIVATE KEY" in lines[0]:
        raise ValueError(
            "The private key is encrypted. This cell expects an unencrypted PEM key. "
            "Either store the password separately and pass it to load_pem_private_key, "
            "or export an unencrypted private key for this demo."
        )

    if "BEGIN OPENSSH PRIVATE KEY" in lines[0]:
        raise ValueError(
            "This is an OpenSSH private key. Store/export the key as PEM instead, "
            "for example BEGIN PRIVATE KEY or BEGIN RSA PRIVATE KEY."
        )

    private_key = serialization.load_pem_private_key(
        pem_text.encode("utf-8"),
        password=None,
    )

    return private_key


# ============================================================
# Load secrets
# ============================================================

API_KEY_ID = get_secret(
    SECRET_SCOPE,
    API_KEY_ID_SECRET,
    required=True
)

private_key = load_kalshi_private_key()

print("Loaded Kalshi API key id length:", len(API_KEY_ID))
print("Loaded Kalshi private key object:", type(private_key).__name__)


# ============================================================
# Signing helpers
# ============================================================

def sign_request(private_key, timestamp_ms: str, method: str, path: str) -> str:
    """
    Kalshi signs:
      timestamp + HTTP_METHOD + path_without_query

    Example:
      1703123456789GET/trade-api/v2/portfolio/balance
    """
    path_without_query = path.split("?")[0]
    message = f"{timestamp_ms}{method.upper()}{path_without_query}".encode("utf-8")

    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.DIGEST_LENGTH,
        ),
        hashes.SHA256(),
    )

    return base64.b64encode(signature).decode("utf-8")


def kalshi_get(path: str, params: dict | None = None, auth: bool = False):
    """
    path should include /trade-api/v2, for example:
      /trade-api/v2/markets
      /trade-api/v2/portfolio/balance
    """
    if not path.startswith(API_PREFIX):
        raise ValueError(f"path must start with {API_PREFIX}. Got: {path}")

    method = "GET"
    url = f"{REST_HOST}{path}"

    headers = {}

    if auth:
        timestamp_ms = str(int(time.time() * 1000))
        signature = sign_request(
            private_key=private_key,
            timestamp_ms=timestamp_ms,
            method=method,
            path=path,
        )

        headers = {
            "KALSHI-ACCESS-KEY": API_KEY_ID,
            "KALSHI-ACCESS-TIMESTAMP": timestamp_ms,
            "KALSHI-ACCESS-SIGNATURE": signature,
        }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=(5, 30),
    )

    print("URL:", response.url)
    print("Status:", response.status_code)

    if response.status_code >= 400:
        print("Response text:", response.text[:1000])
        response.raise_for_status()

    return response.json()


# ============================================================
# Smoke tests
# ============================================================

print("Testing unauthenticated Kalshi market request...")

markets = kalshi_get(
    path=f"{API_PREFIX}/markets",
    params={"limit": 1},
    auth=False,
)

print("Unauthenticated market request succeeded.")
print(markets)

In [0]:
# ============================================================
# Pull 2025 FBS College Football Schedule From ESPN Public API
# ============================================================

import requests
import pandas as pd
from datetime import datetime, timezone


ESPN_CFB_SCOREBOARD_URL = (
    "https://site.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
)

# ESPN group=80 is commonly used for FBS / Division I-A.
ESPN_FBS_GROUP = 80


def fetch_espn_cfb_scoreboard_date_range(
    start_date: str,
    end_date: str,
    group: int = ESPN_FBS_GROUP,
    limit: int = 1000,
) -> dict:
    """
    Fetch ESPN college football scoreboard for a date range.

    start_date/end_date format:
      YYYY-MM-DD

    ESPN dates param format:
      YYYYMMDD-YYYYMMDD
    """
    start = start_date.replace("-", "")
    end = end_date.replace("-", "")

    params = {
        "groups": group,
        "dates": f"{start}-{end}",
        "limit": limit,
    }

    response = requests.get(
        ESPN_CFB_SCOREBOARD_URL,
        params=params,
        timeout=(5, 30),
    )

    print("URL:", response.url)
    print("Status:", response.status_code)

    response.raise_for_status()
    return response.json()


def parse_espn_cfb_games(scoreboard_json: dict) -> pd.DataFrame:
    rows = []

    for event in scoreboard_json.get("events", []):
        competition = (event.get("competitions") or [{}])[0]
        competitors = competition.get("competitors") or []

        home = next((c for c in competitors if c.get("homeAway") == "home"), {})
        away = next((c for c in competitors if c.get("homeAway") == "away"), {})

        home_team = home.get("team") or {}
        away_team = away.get("team") or {}

        rows.append({
            "espn_event_id": event.get("id"),
            "name": event.get("name"),
            "short_name": event.get("shortName"),
            "date_utc": event.get("date"),
            "season_year": (event.get("season") or {}).get("year"),
            "season_type": (event.get("season") or {}).get("type"),
            "week": (event.get("week") or {}).get("number"),
            "status_type": ((competition.get("status") or {}).get("type") or {}).get("name"),
            "completed": ((competition.get("status") or {}).get("type") or {}).get("completed"),
            "neutral_site": competition.get("neutralSite"),
            "conference_competition": competition.get("conferenceCompetition"),
            "home_team": home_team.get("displayName"),
            "home_abbrev_espn": home_team.get("abbreviation"),
            "home_score": home.get("score"),
            "away_team": away_team.get("displayName"),
            "away_abbrev_espn": away_team.get("abbreviation"),
            "away_score": away.get("score"),
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df["date_utc"] = pd.to_datetime(df["date_utc"], utc=True, errors="coerce")
        df = df.sort_values("date_utc")

    return df


# 2025 regular season + bowls/playoff window.
# Adjust as needed.
espn_raw_2025 = fetch_espn_cfb_scoreboard_date_range(
    start_date="2025-08-01",
    end_date="2026-01-31",
)

espn_cfb_games_2025_df = parse_espn_cfb_games(espn_raw_2025)

print("Games returned:", len(espn_cfb_games_2025_df))
display(espn_cfb_games_2025_df)

In [0]:
# ============================================================
# Generate Home + Away Kalshi Ticker Candidates
# ============================================================

import pandas as pd
import re


KALSHI_CFB_GAME_SERIES = "KXNCAAFGAME"


ESPN_TO_KALSHI_ABBR_OVERRIDES = {
    # Publicly verified / common conventions
    "UNC": "UNC",
    "TCU": "TCU",

    # Common ESPN -> Kalshi cleanup
    "SC": "SCAR",
    "NCSU": "NCST",
    "WIS": "WIS",
    "BOIS": "BSU",
    "OU": "OKLA",
    "OKLA": "OKLA",
    "IU": "IND",
    "TA&M": "TXAM",
    "TAMU": "TXAM",
    "NU": "NW",

    # Power / common FBS teams
    "ALA": "ALA",
    "UGA": "UGA",
    "OSU": "OSU",
    "MICH": "MICH",
    "TEX": "TEX",
    "LSU": "LSU",
    "CLEM": "CLEM",
    "ORE": "ORE",
    "ND": "ND",
    "PSU": "PSU",
    "FSU": "FSU",
    "TENN": "TENN",
    "USC": "USC",
    "OKST": "OKST",
    "TTU": "TTU",
    "UCLA": "UCLA",
    "MIA": "MIA",
    "VT": "VT",
    "UVA": "UVA",
    "GT": "GT",
    "PITT": "PITT",
    "WVU": "WVU",
    "MINN": "MINN",
    "IOWA": "IOWA",
    "ISU": "ISU",
    "KSU": "KSU",
    "KU": "KU",
    "UK": "UK",
    "ARK": "ARK",
    "AUB": "AUB",
    "MIZ": "MIZZ",
    "MISS": "MISS",
    "MSST": "MSST",
    "LOU": "LOU",
    "SYR": "SYR",
    "DUKE": "DUKE",
    "STAN": "STAN",
    "CAL": "CAL",
    "COLO": "COLO",
    "UTAH": "UTAH",
    "ASU": "ASU",
    "ARIZ": "ARIZ",
    "BAY": "BAY",
    "UCF": "UCF",
    "HOU": "HOU",
    "CIN": "CIN",
    "BYU": "BYU",
    "SMU": "SMU",
    "NEB": "NEB",
    "MD": "MD",
    "PUR": "PUR",
    "NW": "NW",
    "ILL": "ILL",
    "IND": "IND",
    "RUTG": "RUTG",
    "BC": "BC",
    "WAKE": "WAKE",
    "MEM": "MEM",
    "TULN": "TULN",
    "NAVY": "NAVY",
    "ARMY": "ARMY",
    "AF": "AF",
    "APP": "APP",
    "JMU": "JMU",
    "LIB": "LIB",
    "ODU": "ODU",
    "ECU": "ECU",
    "USF": "USF",
    "FAU": "FAU",
    "FIU": "FIU",
    "UNT": "UNT",
    "UTSA": "UTSA",
    "RICE": "RICE",
    "TULSA": "TULSA",
    "TEM": "TEM",
    "CHAR": "CHAR",
    "CLT": "CHAR",
    "CONN": "CONN",
    "UMASS": "UMASS",
    "HAW": "HAW",
    "SJSU": "SJSU",
    "SDSU": "SDSU",
    "FRES": "FRES",
    "NEV": "NEV",
    "UNLV": "UNLV",
    "USU": "USU",
    "WYO": "WYO",
    "CSU": "CSU",
    "UNM": "UNM",
    "UTEP": "UTEP",
    "NMSU": "NMSU",
    "MTSU": "MTU",
    "WKU": "WKU",
    "LT": "LT",
    "SHSU": "SHSU",
    "JVST": "JVST",
    "KENN": "KENN",
    "MRSH": "MRSH",
    "TROY": "TROY",
    "USA": "USA",
    "UL": "ULL",
    "ULM": "ULM",
    "ARKST": "ARST",
    "ARST": "ARST",
    "TXST": "TXST",
    "GASO": "GASO",
    "GAST": "GAST",
    "CCU": "CCAR",
    "BUFF": "BUFF",
    "AKR": "AKR",
    "KENT": "KENT",
    "BGSU": "BGSU",
    "TOL": "TOL",
    "NIU": "NIU",
    "EMU": "EMU",
    "CMU": "CMU",
    "WMU": "WMU",
    "BALL": "BALL",
    "OHIO": "OHIO",
    "M-OH": "M-OH",
    "MOST": "MOST",
    "NDSU": "NDSU",
    "SAC": "SAC",
}


def date_utc_to_kalshi_date_part(date_utc) -> str:
    ts = pd.to_datetime(date_utc, utc=True)
    eastern_ts = ts.tz_convert("America/New_York")
    return eastern_ts.strftime("%y%b%d").upper()


def clean_abbr_for_kalshi(abbr: str) -> str | None:
    if pd.isna(abbr):
        return None

    abbr = str(abbr).strip().upper()

    if abbr in ESPN_TO_KALSHI_ABBR_OVERRIDES:
        return ESPN_TO_KALSHI_ABBR_OVERRIDES[abbr]

    return re.sub(r"[^A-Z0-9-]", "", abbr)


def generate_side_kalshi_market_ticker_candidates_from_espn_row(row, side: str) -> list[str]:
    """
    Generate market ticker candidates for either the home YES or away YES market.

    side:
      - "home"
      - "away"

    We try both possible event ticker orderings:
      KXNCAAFGAME-25AUG30AWAYHOME-SIDE
      KXNCAAFGAME-25AUG30HOMEAWAY-SIDE
    """
    if side not in ["home", "away"]:
        raise ValueError("side must be either 'home' or 'away'")

    date_part = date_utc_to_kalshi_date_part(row["date_utc"])

    away = clean_abbr_for_kalshi(row["away_abbrev_espn"])
    home = clean_abbr_for_kalshi(row["home_abbrev_espn"])

    if not away or not home:
        return []

    side_abbrev = home if side == "home" else away

    return [
        f"{KALSHI_CFB_GAME_SERIES}-{date_part}{away}{home}-{side_abbrev}",
        f"{KALSHI_CFB_GAME_SERIES}-{date_part}{home}{away}-{side_abbrev}",
    ]


cfb_games_with_candidates_df = espn_cfb_games_2025_df.copy()

cfb_games_with_candidates_df["kalshi_date_part"] = cfb_games_with_candidates_df["date_utc"].apply(
    date_utc_to_kalshi_date_part
)

cfb_games_with_candidates_df["away_abbrev_kalshi"] = cfb_games_with_candidates_df["away_abbrev_espn"].apply(
    clean_abbr_for_kalshi
)

cfb_games_with_candidates_df["home_abbrev_kalshi"] = cfb_games_with_candidates_df["home_abbrev_espn"].apply(
    clean_abbr_for_kalshi
)

cfb_games_with_candidates_df["kalshi_home_market_ticker_candidates"] = cfb_games_with_candidates_df.apply(
    lambda row: generate_side_kalshi_market_ticker_candidates_from_espn_row(row, side="home"),
    axis=1,
)

cfb_games_with_candidates_df["kalshi_away_market_ticker_candidates"] = cfb_games_with_candidates_df.apply(
    lambda row: generate_side_kalshi_market_ticker_candidates_from_espn_row(row, side="away"),
    axis=1,
)

cfb_games_with_candidates_df["kalshi_home_market_candidate_1"] = cfb_games_with_candidates_df[
    "kalshi_home_market_ticker_candidates"
].apply(lambda x: x[0] if len(x) > 0 else None)

cfb_games_with_candidates_df["kalshi_home_market_candidate_2"] = cfb_games_with_candidates_df[
    "kalshi_home_market_ticker_candidates"
].apply(lambda x: x[1] if len(x) > 1 else None)

cfb_games_with_candidates_df["kalshi_away_market_candidate_1"] = cfb_games_with_candidates_df[
    "kalshi_away_market_ticker_candidates"
].apply(lambda x: x[0] if len(x) > 0 else None)

cfb_games_with_candidates_df["kalshi_away_market_candidate_2"] = cfb_games_with_candidates_df[
    "kalshi_away_market_ticker_candidates"
].apply(lambda x: x[1] if len(x) > 1 else None)

display(
    cfb_games_with_candidates_df[
        [
            "espn_event_id",
            "date_utc",
            "week",
            "away_team",
            "away_abbrev_espn",
            "away_abbrev_kalshi",
            "home_team",
            "home_abbrev_espn",
            "home_abbrev_kalshi",
            "kalshi_home_market_candidate_1",
            "kalshi_home_market_candidate_2",
            "kalshi_away_market_candidate_1",
            "kalshi_away_market_candidate_2",
        ]
    ]
)

In [0]:
# Known public current/future example from Kalshi URL:
# North Carolina vs TCU, Aug 29, 2026
# candidate_tickers = generate_kalshi_cfb_game_ticker_candidates(
#     game_date="2026-08-29",
#     team_a="North Carolina",
#     team_b="TCU",
# )

# candidate_tickers

In [0]:
# ============================================================
# Validate Home + Away Kalshi Market Tickers
# ============================================================

import time
import requests
import pandas as pd


TEST_LIMIT = None
PERIOD_INTERVAL = 1440
SLEEP_SECONDS = 0.05
DAYS_BEFORE_GAME = 30
DAYS_AFTER_GAME = 3


def get_candle_window_around_game(
    date_utc,
    days_before: int = DAYS_BEFORE_GAME,
    days_after: int = DAYS_AFTER_GAME,
) -> tuple[int, int]:
    game_ts = pd.to_datetime(date_utc, utc=True)
    start_ts = int((game_ts - pd.Timedelta(days=days_before)).timestamp())
    end_ts = int((game_ts + pd.Timedelta(days=days_after)).timestamp())
    return start_ts, end_ts


def extract_candles_from_response(data: dict) -> list:
    return (
        data.get("candlesticks")
        or data.get("market_candlesticks")
        or data.get("candles")
        or []
    )


def test_historical_candlestick_ticker_quiet(
    ticker: str,
    start_ts: int,
    end_ts: int,
    period_interval: int = PERIOD_INTERVAL,
) -> dict:
    url = f"{REST_HOST}{API_PREFIX}/historical/markets/{ticker}/candlesticks"

    try:
        response = requests.get(
            url,
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": period_interval,
            },
            timeout=(5, 30),
        )

        result = {
            "ticker": ticker,
            "endpoint_type": "historical",
            "url": response.url,
            "status_code": response.status_code,
            "exists": response.status_code == 200,
            "candles_count": 0,
            "response_preview": response.text[:500],
        }

        if response.status_code == 200:
            data = response.json()
            candles = extract_candles_from_response(data)
            result["candles_count"] = len(candles)

        return result

    except Exception as e:
        return {
            "ticker": ticker,
            "endpoint_type": "historical",
            "url": url,
            "status_code": None,
            "exists": False,
            "candles_count": 0,
            "response_preview": repr(e),
        }


def test_live_candlestick_ticker_quiet(
    ticker: str,
    start_ts: int,
    end_ts: int,
    period_interval: int = PERIOD_INTERVAL,
    series_ticker: str = KALSHI_CFB_GAME_SERIES,
) -> dict:
    url = f"{REST_HOST}{API_PREFIX}/series/{series_ticker}/markets/{ticker}/candlesticks"

    try:
        response = requests.get(
            url,
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": period_interval,
            },
            timeout=(5, 30),
        )

        result = {
            "ticker": ticker,
            "endpoint_type": "live_recent",
            "url": response.url,
            "status_code": response.status_code,
            "exists": response.status_code == 200,
            "candles_count": 0,
            "response_preview": response.text[:500],
        }

        if response.status_code == 200:
            data = response.json()
            candles = extract_candles_from_response(data)
            result["candles_count"] = len(candles)

        return result

    except Exception as e:
        return {
            "ticker": ticker,
            "endpoint_type": "live_recent",
            "url": url,
            "status_code": None,
            "exists": False,
            "candles_count": 0,
            "response_preview": repr(e),
        }


def find_valid_side_kalshi_ticker_for_game(
    row,
    side: str,
    sleep_seconds: float = SLEEP_SECONDS,
    try_live_fallback: bool = True,
) -> dict:
    """
    Validate one side of a game:
      side="home" -> home YES market
      side="away" -> away YES market
    """
    if side not in ["home", "away"]:
        raise ValueError("side must be either 'home' or 'away'")

    if side == "home":
        candidates = row.get("kalshi_home_market_ticker_candidates") or []
        market_side = "home_yes"
        selection_team = row["home_team"]
        selection_abbrev_kalshi = row["home_abbrev_kalshi"]
        opponent_team = row["away_team"]
        opponent_abbrev_kalshi = row["away_abbrev_kalshi"]
    else:
        candidates = row.get("kalshi_away_market_ticker_candidates") or []
        market_side = "away_yes"
        selection_team = row["away_team"]
        selection_abbrev_kalshi = row["away_abbrev_kalshi"]
        opponent_team = row["home_team"]
        opponent_abbrev_kalshi = row["home_abbrev_kalshi"]

    start_ts, end_ts = get_candle_window_around_game(row["date_utc"])
    attempts = []

    for candidate in candidates:
        historical_result = test_historical_candlestick_ticker_quiet(
            ticker=candidate,
            start_ts=start_ts,
            end_ts=end_ts,
            period_interval=PERIOD_INTERVAL,
        )

        attempts.append(historical_result)

        if sleep_seconds:
            time.sleep(sleep_seconds)

        if historical_result["exists"]:
            return {
                "espn_event_id": row["espn_event_id"],
                "date_utc": row["date_utc"],
                "week": row["week"],
                "away_team": row["away_team"],
                "away_abbrev_kalshi": row["away_abbrev_kalshi"],
                "home_team": row["home_team"],
                "home_abbrev_kalshi": row["home_abbrev_kalshi"],
                "market_side": market_side,
                "selection_team": selection_team,
                "selection_abbrev_kalshi": selection_abbrev_kalshi,
                "opponent_team": opponent_team,
                "opponent_abbrev_kalshi": opponent_abbrev_kalshi,
                "kalshi_ticker_found": True,
                "kalshi_ticker": candidate,
                "kalshi_endpoint_type": historical_result["endpoint_type"],
                "kalshi_validation_status_code": historical_result["status_code"],
                "kalshi_candles_count": historical_result["candles_count"],
                "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
                "kalshi_attempt_results": attempts,
            }

        if try_live_fallback:
            live_result = test_live_candlestick_ticker_quiet(
                ticker=candidate,
                start_ts=start_ts,
                end_ts=end_ts,
                period_interval=PERIOD_INTERVAL,
                series_ticker=KALSHI_CFB_GAME_SERIES,
            )

            attempts.append(live_result)

            if sleep_seconds:
                time.sleep(sleep_seconds)

            if live_result["exists"]:
                return {
                    "espn_event_id": row["espn_event_id"],
                    "date_utc": row["date_utc"],
                    "week": row["week"],
                    "away_team": row["away_team"],
                    "away_abbrev_kalshi": row["away_abbrev_kalshi"],
                    "home_team": row["home_team"],
                    "home_abbrev_kalshi": row["home_abbrev_kalshi"],
                    "market_side": market_side,
                    "selection_team": selection_team,
                    "selection_abbrev_kalshi": selection_abbrev_kalshi,
                    "opponent_team": opponent_team,
                    "opponent_abbrev_kalshi": opponent_abbrev_kalshi,
                    "kalshi_ticker_found": True,
                    "kalshi_ticker": candidate,
                    "kalshi_endpoint_type": live_result["endpoint_type"],
                    "kalshi_validation_status_code": live_result["status_code"],
                    "kalshi_candles_count": live_result["candles_count"],
                    "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
                    "kalshi_attempt_results": attempts,
                }

    return {
        "espn_event_id": row["espn_event_id"],
        "date_utc": row["date_utc"],
        "week": row["week"],
        "away_team": row["away_team"],
        "away_abbrev_kalshi": row["away_abbrev_kalshi"],
        "home_team": row["home_team"],
        "home_abbrev_kalshi": row["home_abbrev_kalshi"],
        "market_side": market_side,
        "selection_team": selection_team,
        "selection_abbrev_kalshi": selection_abbrev_kalshi,
        "opponent_team": opponent_team,
        "opponent_abbrev_kalshi": opponent_abbrev_kalshi,
        "kalshi_ticker_found": False,
        "kalshi_ticker": None,
        "kalshi_endpoint_type": None,
        "kalshi_validation_status_code": None,
        "kalshi_candles_count": 0,
        "kalshi_attempted_tickers": [a["ticker"] for a in attempts],
        "kalshi_attempt_results": attempts,
    }


source_df = cfb_games_with_candidates_df.copy()

if TEST_LIMIT is not None:
    source_df = source_df.head(TEST_LIMIT)

validation_rows = []

for idx, row in source_df.iterrows():
    game_num = (len(validation_rows) // 2) + 1

    print(
        f"[{game_num}/{len(source_df)}] "
        f"{row['away_team']} at {row['home_team']} "
        f"({row['date_utc']})"
    )

    home_validation = find_valid_side_kalshi_ticker_for_game(
        row=row,
        side="home",
        sleep_seconds=SLEEP_SECONDS,
        try_live_fallback=True,
    )

    away_validation = find_valid_side_kalshi_ticker_for_game(
        row=row,
        side="away",
        sleep_seconds=SLEEP_SECONDS,
        try_live_fallback=True,
    )

    validation_rows.append(home_validation)
    validation_rows.append(away_validation)

    print("  HOME:", home_validation["kalshi_ticker_found"], home_validation["kalshi_ticker"])
    print("  AWAY:", away_validation["kalshi_ticker_found"], away_validation["kalshi_ticker"])

kalshi_side_ticker_validation_df = pd.DataFrame(validation_rows)

print("\nValidation complete.")
print("Games checked:", len(source_df))
print("Side rows checked:", len(kalshi_side_ticker_validation_df))
print("Tickers found:", kalshi_side_ticker_validation_df["kalshi_ticker_found"].fillna(False).sum())

display(kalshi_side_ticker_validation_df)

In [0]:
# ============================================================
# Join Home + Away Validated Tickers Back to Game Table
# ============================================================

import pandas as pd


home_validated_df = kalshi_side_ticker_validation_df[
    kalshi_side_ticker_validation_df["market_side"] == "home_yes"
][
    [
        "espn_event_id",
        "kalshi_ticker_found",
        "kalshi_ticker",
        "kalshi_endpoint_type",
        "kalshi_validation_status_code",
        "kalshi_candles_count",
        "kalshi_attempted_tickers",
        "kalshi_attempt_results",
    ]
].rename(
    columns={
        "kalshi_ticker_found": "kalshi_home_ticker_found",
        "kalshi_ticker": "kalshi_home_ticker",
        "kalshi_endpoint_type": "kalshi_home_endpoint_type",
        "kalshi_validation_status_code": "kalshi_home_validation_status_code",
        "kalshi_candles_count": "kalshi_home_candles_count",
        "kalshi_attempted_tickers": "kalshi_home_attempted_tickers",
        "kalshi_attempt_results": "kalshi_home_attempt_results",
    }
)

away_validated_df = kalshi_side_ticker_validation_df[
    kalshi_side_ticker_validation_df["market_side"] == "away_yes"
][
    [
        "espn_event_id",
        "kalshi_ticker_found",
        "kalshi_ticker",
        "kalshi_endpoint_type",
        "kalshi_validation_status_code",
        "kalshi_candles_count",
        "kalshi_attempted_tickers",
        "kalshi_attempt_results",
    ]
].rename(
    columns={
        "kalshi_ticker_found": "kalshi_away_ticker_found",
        "kalshi_ticker": "kalshi_away_ticker",
        "kalshi_endpoint_type": "kalshi_away_endpoint_type",
        "kalshi_validation_status_code": "kalshi_away_validation_status_code",
        "kalshi_candles_count": "kalshi_away_candles_count",
        "kalshi_attempted_tickers": "kalshi_away_attempted_tickers",
        "kalshi_attempt_results": "kalshi_away_attempt_results",
    }
)

cfb_games_home_away_kalshi_ticker_df = (
    cfb_games_with_candidates_df
    .merge(home_validated_df, on="espn_event_id", how="left")
    .merge(away_validated_df, on="espn_event_id", how="left")
)

cfb_games_home_away_kalshi_ticker_df["kalshi_home_ticker_found"] = (
    cfb_games_home_away_kalshi_ticker_df["kalshi_home_ticker_found"]
    .fillna(False)
    .astype(bool)
)

cfb_games_home_away_kalshi_ticker_df["kalshi_away_ticker_found"] = (
    cfb_games_home_away_kalshi_ticker_df["kalshi_away_ticker_found"]
    .fillna(False)
    .astype(bool)
)

cfb_games_home_away_kalshi_ticker_df["both_side_tickers_found"] = (
    cfb_games_home_away_kalshi_ticker_df["kalshi_home_ticker_found"]
    & cfb_games_home_away_kalshi_ticker_df["kalshi_away_ticker_found"]
)

print("Total games:", len(cfb_games_home_away_kalshi_ticker_df))
print("Home tickers found:", cfb_games_home_away_kalshi_ticker_df["kalshi_home_ticker_found"].sum())
print("Away tickers found:", cfb_games_home_away_kalshi_ticker_df["kalshi_away_ticker_found"].sum())
print("Both-side tickers found:", cfb_games_home_away_kalshi_ticker_df["both_side_tickers_found"].sum())

display(
    cfb_games_home_away_kalshi_ticker_df[
        [
            "espn_event_id",
            "date_utc",
            "week",
            "away_team",
            "away_abbrev_espn",
            "away_abbrev_kalshi",
            "home_team",
            "home_abbrev_espn",
            "home_abbrev_kalshi",
            "kalshi_home_ticker_found",
            "kalshi_home_ticker",
            "kalshi_home_endpoint_type",
            "kalshi_away_ticker_found",
            "kalshi_away_ticker",
            "kalshi_away_endpoint_type",
            "both_side_tickers_found",
        ]
    ]
)

In [0]:
# ============================================================
# Build Long Home/Away Ticker Universe for Trade Landing
# ============================================================

import pandas as pd


valid_game_rows = cfb_games_home_away_kalshi_ticker_df.copy()

home_ticker_universe_df = valid_game_rows[
    valid_game_rows["kalshi_home_ticker_found"] == True
].copy()

home_ticker_universe_df["market_side"] = "home_yes"
home_ticker_universe_df["selection_team"] = home_ticker_universe_df["home_team"]
home_ticker_universe_df["selection_abbrev_kalshi"] = home_ticker_universe_df["home_abbrev_kalshi"]
home_ticker_universe_df["opponent_team"] = home_ticker_universe_df["away_team"]
home_ticker_universe_df["opponent_abbrev_kalshi"] = home_ticker_universe_df["away_abbrev_kalshi"]
home_ticker_universe_df["kalshi_ticker"] = home_ticker_universe_df["kalshi_home_ticker"]
home_ticker_universe_df["kalshi_endpoint_type"] = home_ticker_universe_df["kalshi_home_endpoint_type"]

away_ticker_universe_df = valid_game_rows[
    valid_game_rows["kalshi_away_ticker_found"] == True
].copy()

away_ticker_universe_df["market_side"] = "away_yes"
away_ticker_universe_df["selection_team"] = away_ticker_universe_df["away_team"]
away_ticker_universe_df["selection_abbrev_kalshi"] = away_ticker_universe_df["away_abbrev_kalshi"]
away_ticker_universe_df["opponent_team"] = away_ticker_universe_df["home_team"]
away_ticker_universe_df["opponent_abbrev_kalshi"] = away_ticker_universe_df["home_abbrev_kalshi"]
away_ticker_universe_df["kalshi_ticker"] = away_ticker_universe_df["kalshi_away_ticker"]
away_ticker_universe_df["kalshi_endpoint_type"] = away_ticker_universe_df["kalshi_away_endpoint_type"]

kalshi_cfb_trade_ticker_universe_df = pd.concat(
    [home_ticker_universe_df, away_ticker_universe_df],
    ignore_index=True,
)

kalshi_cfb_trade_ticker_universe_df = kalshi_cfb_trade_ticker_universe_df[
    [
        "espn_event_id",
        "date_utc",
        "week",
        "away_team",
        "away_abbrev_espn",
        "away_abbrev_kalshi",
        "away_score",
        "home_team",
        "home_abbrev_espn",
        "home_abbrev_kalshi",
        "home_score",
        "market_side",
        "selection_team",
        "selection_abbrev_kalshi",
        "opponent_team",
        "opponent_abbrev_kalshi",
        "kalshi_ticker",
        "kalshi_endpoint_type",
    ]
].copy()

kalshi_cfb_trade_ticker_universe_df["home_score_num"] = pd.to_numeric(
    kalshi_cfb_trade_ticker_universe_df["home_score"],
    errors="coerce",
)

kalshi_cfb_trade_ticker_universe_df["away_score_num"] = pd.to_numeric(
    kalshi_cfb_trade_ticker_universe_df["away_score"],
    errors="coerce",
)

kalshi_cfb_trade_ticker_universe_df["home_team_won"] = (
    kalshi_cfb_trade_ticker_universe_df["home_score_num"]
    > kalshi_cfb_trade_ticker_universe_df["away_score_num"]
)

kalshi_cfb_trade_ticker_universe_df["selection_team_won"] = (
    (
        (kalshi_cfb_trade_ticker_universe_df["market_side"] == "home_yes")
        & kalshi_cfb_trade_ticker_universe_df["home_team_won"]
    )
    |
    (
        (kalshi_cfb_trade_ticker_universe_df["market_side"] == "away_yes")
        & (~kalshi_cfb_trade_ticker_universe_df["home_team_won"])
    )
)

print("Trade ticker universe rows:", len(kalshi_cfb_trade_ticker_universe_df))
print("Unique games:", kalshi_cfb_trade_ticker_universe_df["espn_event_id"].nunique())
print("Unique tickers:", kalshi_cfb_trade_ticker_universe_df["kalshi_ticker"].nunique())
print("Home side rows:", (kalshi_cfb_trade_ticker_universe_df["market_side"] == "home_yes").sum())
print("Away side rows:", (kalshi_cfb_trade_ticker_universe_df["market_side"] == "away_yes").sum())

display(kalshi_cfb_trade_ticker_universe_df)

In [0]:
# ============================================================
# Create Unity Catalog Objects for Landing Volume
# ============================================================

CATALOG = "databricks_realtime_optimization"
SCHEMA = "cfb_analytics"
VOLUME = "landing"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

print("Created or confirmed:")
print(f"Catalog: {CATALOG}")
print(f"Schema:  {CATALOG}.{SCHEMA}")
print(f"Volume:  {CATALOG}.{SCHEMA}.{VOLUME}")

display(
    spark.sql(f"DESCRIBE VOLUME {CATALOG}.{SCHEMA}.{VOLUME}")
)

In [0]:
# ============================================================
# Initial Trades JSON Landing Ingest
# ============================================================

import json
import time
import uuid
import requests
import pandas as pd
from datetime import datetime, timezone


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

SEASON = 2025

# IMPORTANT:
# Because the Volume is already named "landing", do NOT add another /landing folder here.
LANDING_BASE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/kalshi/cfb/trades"
)

# Start small while testing. Change to None for all tickers.
TEST_LIMIT = None

LIMIT_PER_PAGE = 1000
SLEEP_SECONDS = 0.05

DAYS_BEFORE_GAME = 7
HOURS_AFTER_GAME = 6

INGEST_RUN_ID = str(uuid.uuid4())
RETRIEVED_AT_UTC = datetime.now(timezone.utc).isoformat()


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_path_value(value) -> str:
    """
    Make values safe for partition-style path segments.
    """
    if pd.isna(value):
        return "unknown"

    text = str(value)
    text = text.replace("/", "_")
    text = text.replace(" ", "_")
    text = text.replace(":", "_")
    text = text.replace("?", "_")
    text = text.replace("&", "_")
    text = text.replace("=", "_")
    text = text.replace("\\", "_")

    return text


def get_trade_window_around_game(
    date_utc,
    days_before: int = DAYS_BEFORE_GAME,
    hours_after: int = HOURS_AFTER_GAME,
) -> tuple[int, int]:
    """
    Trade ingest window:
      start = kickoff - 7 days
      end   = kickoff + 6 hours
    """
    game_ts = pd.to_datetime(date_utc, utc=True)

    min_ts = int((game_ts - pd.Timedelta(days=days_before)).timestamp())
    max_ts = int((game_ts + pd.Timedelta(hours=hours_after)).timestamp())

    return min_ts, max_ts


def get_trade_endpoint_path(endpoint_type: str) -> str:
    """
    endpoint_type is inherited from ticker validation:
      historical  -> /historical/trades
      live_recent -> /markets/trades
    """
    if endpoint_type == "historical":
        return f"{API_PREFIX}/historical/trades"

    if endpoint_type == "live_recent":
        return f"{API_PREFIX}/markets/trades"

    raise ValueError(f"Unknown endpoint_type: {endpoint_type}")


def request_trades_page(
    ticker: str,
    endpoint_type: str,
    min_ts: int,
    max_ts: int,
    limit: int = LIMIT_PER_PAGE,
    cursor: str | None = None,
) -> dict:
    """
    Request one page from the Kalshi trades API.
    """
    path = get_trade_endpoint_path(endpoint_type)
    url = f"{REST_HOST}{path}"

    params = {
        "ticker": ticker,
        "min_ts": min_ts,
        "max_ts": max_ts,
        "limit": limit,
    }

    if cursor:
        params["cursor"] = cursor

    response = requests.get(
        url,
        params=params,
        timeout=(5, 30),
    )

    try:
        payload = response.json() if response.text else None
    except Exception:
        payload = None

    return {
        "url": response.url,
        "status_code": response.status_code,
        "payload": payload,
        "response_text_preview": response.text[:1000],
    }


def build_landing_partition_path(row) -> str:
    """
    One ticker-side partition.

    Example:
      /Volumes/.../landing/kalshi/cfb/trades/
        season=2025/
          espn_event_id=401756846/
            market_side=home_yes/
              ticker=KXNCAAFGAME-25AUG23ISUKSU-KSU/
    """
    return (
        f"{LANDING_BASE_PATH}"
        f"/season={SEASON}"
        f"/espn_event_id={safe_path_value(row['espn_event_id'])}"
        f"/market_side={safe_path_value(row['market_side'])}"
        f"/ticker={safe_path_value(row['kalshi_ticker'])}"
    )


def delete_existing_partition(partition_path: str) -> None:
    """
    Idempotent partition rewrite for testing.
    Deletes old JSON pages before writing new ones.

    Safety checks prevent accidental deletion of the base Volume/folder.
    """
    if partition_path.rstrip("/") == LANDING_BASE_PATH.rstrip("/"):
        raise ValueError("Safety stop: refusing to delete the landing base path.")

    if "/ticker=" not in partition_path:
        raise ValueError(f"Safety stop: partition path does not include ticker=: {partition_path}")

    if not partition_path.startswith(LANDING_BASE_PATH):
        raise ValueError(
            f"Safety stop: partition path is outside landing base path.\n"
            f"partition_path={partition_path}\n"
            f"LANDING_BASE_PATH={LANDING_BASE_PATH}"
        )

    print("  deleting old ticker partition if it exists...")

    try:
        dbutils.fs.rm(partition_path, recurse=True)
        print("  old ticker partition deleted or did not exist.")
    except Exception as e:
        print("  delete skipped or partition did not exist.")
        print("  delete message:", repr(e))


def write_json_to_volume(path: str, obj: dict) -> None:
    """
    Write one JSON object to a file path in a Databricks Volume.
    Ensures the parent directory exists first.
    """
    parent_path = path.rsplit("/", 1)[0]

    dbutils.fs.mkdirs(parent_path)

    dbutils.fs.put(
        path,
        json.dumps(obj, ensure_ascii=False, default=str),
        overwrite=True,
    )


# ------------------------------------------------------------
# Validate required input
# ------------------------------------------------------------

required_columns = [
    "espn_event_id",
    "date_utc",
    "market_side",
    "kalshi_ticker",
    "kalshi_endpoint_type",
]

missing_columns = [
    col for col in required_columns
    if col not in kalshi_cfb_trade_ticker_universe_df.columns
]

if missing_columns:
    raise ValueError(
        f"kalshi_cfb_trade_ticker_universe_df is missing required columns: {missing_columns}"
    )


# ------------------------------------------------------------
# Run landing ingest
# ------------------------------------------------------------

source_df = kalshi_cfb_trade_ticker_universe_df.copy()

if TEST_LIMIT is not None:
    source_df = source_df.head(TEST_LIMIT)

diagnostic_rows = []

print("Ingest run id:", INGEST_RUN_ID)
print("Landing base path:", LANDING_BASE_PATH)
print("Ticker rows to ingest:", len(source_df))

for ticker_index, (_, row) in enumerate(source_df.iterrows(), start=1):
    ticker = row["kalshi_ticker"]
    endpoint_type = row["kalshi_endpoint_type"]
    market_side = row["market_side"]
    espn_event_id = row["espn_event_id"]

    min_ts, max_ts = get_trade_window_around_game(row["date_utc"])
    partition_path = build_landing_partition_path(row)

    print(
        f"\n[{ticker_index}/{len(source_df)}] "
        f"{market_side} | {ticker} | {row.get('away_team')} at {row.get('home_team')}"
    )
    print("Partition:", partition_path)
    print("Window min_ts:", min_ts)
    print("Window max_ts:", max_ts)
    print("Endpoint type:", endpoint_type)

    delete_existing_partition(partition_path)

    cursor = None
    page_num = 0
    total_trades_for_ticker = 0
    ticker_failed = False

    while True:
        page_num += 1

        print(f"  requesting trades page {page_num}...")

        result = request_trades_page(
            ticker=ticker,
            endpoint_type=endpoint_type,
            min_ts=min_ts,
            max_ts=max_ts,
            limit=LIMIT_PER_PAGE,
            cursor=cursor,
        )

        payload = result["payload"] or {}
        trades = payload.get("trades") or []
        next_cursor = payload.get("cursor")

        page_obj = {
            "ingest_run_id": INGEST_RUN_ID,
            "retrieved_at_utc": RETRIEVED_AT_UTC,
            "season": SEASON,

            "espn_event_id": espn_event_id,
            "date_utc": str(row.get("date_utc")),
            "week": row.get("week"),

            "away_team": row.get("away_team"),
            "away_abbrev_espn": row.get("away_abbrev_espn"),
            "away_abbrev_kalshi": row.get("away_abbrev_kalshi"),
            "away_score": row.get("away_score"),

            "home_team": row.get("home_team"),
            "home_abbrev_espn": row.get("home_abbrev_espn"),
            "home_abbrev_kalshi": row.get("home_abbrev_kalshi"),
            "home_score": row.get("home_score"),

            "market_side": market_side,
            "selection_team": row.get("selection_team"),
            "selection_abbrev_kalshi": row.get("selection_abbrev_kalshi"),
            "opponent_team": row.get("opponent_team"),
            "opponent_abbrev_kalshi": row.get("opponent_abbrev_kalshi"),

            "kalshi_ticker": ticker,
            "kalshi_endpoint_type": endpoint_type,

            "min_ts": min_ts,
            "max_ts": max_ts,
            "limit": LIMIT_PER_PAGE,
            "page_num": page_num,

            "request_url": result["url"],
            "status_code": result["status_code"],
            "trade_count_on_page": len(trades),

            "response": payload,
        }

        page_path = f"{partition_path}/page_{page_num:06d}.json"

        print("  writing page:", page_path)

        write_json_to_volume(page_path, page_obj)

        diagnostic_rows.append(
            {
                "ingest_run_id": INGEST_RUN_ID,
                "retrieved_at_utc": RETRIEVED_AT_UTC,
                "season": SEASON,
                "espn_event_id": espn_event_id,
                "date_utc": str(row.get("date_utc")),
                "market_side": market_side,
                "kalshi_ticker": ticker,
                "kalshi_endpoint_type": endpoint_type,
                "min_ts": min_ts,
                "max_ts": max_ts,
                "page_num": page_num,
                "status_code": result["status_code"],
                "trade_count_on_page": len(trades),
                "request_url": result["url"],
                "page_path": page_path,
                "response_text_preview": result["response_text_preview"],
            }
        )

        print(
            f"  page={page_num} "
            f"status={result['status_code']} "
            f"trades={len(trades)}"
        )

        if result["status_code"] != 200:
            ticker_failed = True
            print("  WARNING: non-200 response. Stopping this ticker.")
            print("  response preview:", result["response_text_preview"])
            break

        total_trades_for_ticker += len(trades)

        if not next_cursor:
            break

        cursor = next_cursor

        if SLEEP_SECONDS:
            time.sleep(SLEEP_SECONDS)

    print("  total trades for ticker:", total_trades_for_ticker)

    if ticker_failed:
        print("  WARNING: ticker ingest failed before all pages completed")

    if SLEEP_SECONDS:
        time.sleep(SLEEP_SECONDS)


kalshi_cfb_trades_landing_diagnostics_df = pd.DataFrame(diagnostic_rows)

print("\nLanding ingest complete.")
print("Pages written:", len(kalshi_cfb_trades_landing_diagnostics_df))

if not kalshi_cfb_trades_landing_diagnostics_df.empty:
    print(
        "Total trades reported on pages:",
        kalshi_cfb_trades_landing_diagnostics_df["trade_count_on_page"].sum()
    )

display(kalshi_cfb_trades_landing_diagnostics_df)